In [ ]:
# 설치
!pip install -q transformers==4.36.2 tokenizers==0.15.2 "huggingface-hub<1.0" \
    torchlibrosa==0.1.0 librosa==0.10.2.post1 ruamel.yaml==0.17.40 \
    gdown==5.2.0 "kagglehub>=0.3.12" pycocoevalcap==1.2 \
    "scikit-learn>=1.4,<1.7" "pandas>=2.1,<2.4" tqdm loguru warmup-scheduler gensim

# 저장소 다운로드
!git clone -q --depth 1 --branch working https://github.com/youhan200203/OnomaHoW.git /content/OnomaHoW
!git clone -q https://github.com/XinhaoMei/WavCaps.git /content/WavCaps
!git clone -q https://github.com/jspirit01/sound-to-onomatopoeia.git /content/sound-to-onomatopoeia

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.8/126.8 kB 11.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 155.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 120.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.1/260.1 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.7/113.7 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.3/104.3 MB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 95.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.2/788.2 kB 49.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following depende

In [ ]:
# 설정
import csv
import gc
import json
import math
import os
import random
import re
import shutil
import sys
import unicodedata
from pathlib import Path

import librosa
import kagglehub
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import Audio, display
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import get_linear_schedule_with_warmup

from google.colab import drive

ONOMAHOW_DIR = Path("/content/OnomaHoW")
if str(ONOMAHOW_DIR) not in sys.path:
    sys.path.insert(0, str(ONOMAHOW_DIR))

from jamo_preprocessing import (
    EXPECTED_LATIN_AUDIO,
    EXPECTED_OUTPUT_ROWS,
    load_or_create_split_manifest,
    prepare_rows,
)

drive.mount("/content/drive")

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
WAVCAPS_DIR = "/content/WavCaps"
ANNOTATION_DIR = "/content/sound-to-onomatopoeia"

audio_root = Path(kagglehub.dataset_download("buraktaci/firat-esc50"))
csv_path = Path(ANNOTATION_DIR) / "sound-to-onomatopoeia_annotation.csv"
with csv_path.open(encoding="utf-8-sig", newline="") as stream:
    prepared_rows, processing_report = prepare_rows(csv.DictReader(stream))

assert processing_report.output_rows == EXPECTED_OUTPUT_ROWS == 7_961
assert processing_report.latin_rows_excluded == (EXPECTED_LATIN_AUDIO,)
df = pd.DataFrame(prepared_rows)

100%|██████████| 1.23G/1.23G [01:02<00:00, 21.3MB/s]

Extracting files...


In [ ]:
MODEL_SEED = 20
SPLIT_SEED = 20
EXPERIMENT_NAME = "onomacap_base_v1"
SEED = MODEL_SEED

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

In [ ]:
def key(name):
    return unicodedata.normalize("NFKC", Path(str(name)).name).strip().casefold()

references = [f"candidate{i}_en" for i in range(1, 6)]

for column in references:
    df[column] = (
        df[column]
        .str.replace(r'[,.!?;:|*"]', " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

assert len(df) == 7_961

extensions = {".mp3", ".wav", ".flac", ".ogg", ".m4a"}
audio_files = {
    key(path.name): path
    for path in audio_root.rglob("*")
    if path.suffix.lower() in extensions
}

df["audio_path"] = df["audio_file"].map(
    lambda name: str(audio_files.get(key(name), ""))
)
assert len(df) == 7_961
assert df["class"].nunique() == 41
assert not df["audio_file"].duplicated().any()
assert not df[references].isna().any().any()
assert not df["audio_path"].eq("").any()

In [ ]:
SPLIT_MANIFEST = Path(
    "/content/drive/MyDrive/OnomaCap/splits/"
    "onomacap_7961_seed20_v1.csv"
)
splits = load_or_create_split_manifest(
    df,
    SPLIT_MANIFEST,
    split_seed=SPLIT_SEED,
)

assert len(splits["train"]) == 6_368
assert len(splits["val"]) == 796
assert len(splits["test"]) == 797

print({name: len(frame) for name, frame in splits.items()})
print("split seed:", SPLIT_SEED, "model seed:", MODEL_SEED)

In [ ]:
JSON_DIR = Path(
    "/content/WavCaps/captioning/data/OnomaCap/json_files"
)
JSON_DIR.mkdir(parents=True, exist_ok=True)


def to_wavcaps_item(row):
    return {
        "audio": str(Path(row["audio_path"]).resolve()),
        "caption_1": str(row["candidate1_en"]),
        "caption_2": str(row["candidate2_en"]),
        "caption_3": str(row["candidate3_en"]),
        "caption_4": str(row["candidate4_en"]),
        "caption_5": str(row["candidate5_en"]),
    }


for split_name, frame in splits.items():
    output_path = JSON_DIR / f"{split_name}.json"

    records = [
        to_wavcaps_item(row)
        for _, row in frame.iterrows()
    ]

    output_path.write_text(
        json.dumps(
            {"data": records},
            ensure_ascii=False,
            indent=2,
        )
    )

    print(output_path, len(records))

/content/WavCaps/captioning/data/OnomaCap/json_files/train.json 6369
/content/WavCaps/captioning/data/OnomaCap/json_files/val.json 796
/content/WavCaps/captioning/data/OnomaCap/json_files/test.json 797


In [ ]:
%%writefile /content/WavCaps/captioning/settings/onomacap.yaml
exp_name: "onomacap_base_v1"
device: "cuda"
pretrain: false
seed: 20

audio_args:
  sr: 32000
  n_fft: 1024
  hop_length: 320
  f_min: 50
  f_max: 14000
  n_mels: 64
  max_length: 0
  mono: true

data_args:
  dataset: "OnomaCap"
  batch_size: 16
  num_workers: 4

audio_encoder_args:
  model_arch: "transformer"
  model_name: "htsat"
  pretrained: true
  freeze: false
  spec_augment: true

text_decoder_args:
  name: "facebook/bart-base"
  pretrained: false

optim_args:
  scheduler: "cosine"
  lr: !!float 3e-5
  optimizer_name: "adamw"
  betas: [0.9, 0.999]
  eps: !!float 1e-8
  momentum: 0.9
  gamma: 0.1
  warmup_epochs: 1
  step_epochs: 10
  weight_decay: !!float 1e-6

training:
  epochs: 20
  clip_grad: 2
  dropout: 0.2
  precision: "bf16"

Writing /content/WavCaps/captioning/settings/onomacap.yaml


In [ ]:
from pathlib import Path

train_file = Path("/content/WavCaps/captioning/train.py")
train_text = train_file.read_text()

if "ONOMACAP_EPOCH_RESUME" not in train_text:
    train_text = train_text.replace(
        "import platform\nimport argparse\n",
        "import platform\nimport argparse\nimport random\n"
        "from pathlib import Path\nimport numpy as np\n",
        1,
    )

    setup_marker = "    main_logger.info(f'Size of training set:"
    assert setup_marker in train_text
    resume_setup = '''    # ONOMACAP_EPOCH_RESUME
    epoch_checkpoint_dir = (
        Path("/content/drive/MyDrive/OnomaCap/checkpoints")
        / folder_name
        / "epochs"
    )
    epoch_checkpoint_dir.mkdir(parents=True, exist_ok=True)
    best_model_path = epoch_checkpoint_dir.parent / "best_model.pt"

    def load_checkpoint(path):
        try:
            return torch.load(path, map_location=device, weights_only=False)
        except TypeError:
            return torch.load(path, map_location=device)

    def atomic_torch_save(state, path):
        temporary_path = path.with_suffix(path.suffix + ".tmp")
        torch.save(state, temporary_path)
        os.replace(temporary_path, path)

    start_epoch = 1
    loss_stats = []
    spiders = []
    epoch_checkpoints = [
        path for path in epoch_checkpoint_dir.glob("epoch_*.pt")
        if path.stem.removeprefix("epoch_").isdigit()
    ]
    resume_path = max(
        epoch_checkpoints,
        key=lambda path: int(path.stem.removeprefix("epoch_")),
        default=None,
    )

    if resume_path is not None:
        checkpoint = load_checkpoint(resume_path)
        model.load_state_dict(checkpoint["model"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        start_epoch = checkpoint["epoch"] + 1
        loss_stats = checkpoint.get("loss_stats", [])
        spiders = checkpoint.get("spiders", [])
        if "python_rng_state" in checkpoint:
            random.setstate(checkpoint["python_rng_state"])
        if "numpy_rng_state" in checkpoint:
            np.random.set_state(checkpoint["numpy_rng_state"])
        if "torch_rng_state" in checkpoint:
            torch.set_rng_state(checkpoint["torch_rng_state"])
        if torch.cuda.is_available() and checkpoint.get("cuda_rng_state") is not None:
            torch.cuda.set_rng_state_all(checkpoint["cuda_rng_state"])
        main_logger.info(
            f"Resume: {resume_path} (next epoch: {start_epoch})"
        )
    else:
        main_logger.info("No epoch checkpoint found; start at epoch 1.")

'''
    train_text = train_text.replace(
        setup_marker,
        resume_setup + setup_marker,
        1,
    )

    old_loop = '''    # training loop
    loss_stats = []
    spiders = []

    for epoch in range(1, config["training"]["epochs"] + 1):'''
    new_loop = '''    # training loop
    for epoch in range(start_epoch, config["training"]["epochs"] + 1):'''
    assert old_loop in train_text
    train_text = train_text.replace(old_loop, new_loop, 1)

    old_best = '''                if spider >= max(spiders):
                    torch.save({
                        "model": model.state_dict(),
                        "optimizer": optimizer.state_dict(),
                        "beam_size": i,
                        "epoch": epoch,
                        "config": config,
                    }, str(model_output_dir) + '/best_model.pt'.format(epoch))'''
    new_best = '''                if spider >= max(spiders):
                    best_state = {
                        "model": model.state_dict(),
                        "optimizer": optimizer.state_dict(),
                        "beam_size": i,
                        "epoch": epoch,
                        "selection_metric": "spider",
                        "selection_score": float(spider),
                        "val_scores": {
                            name: float(values["score"])
                            for name, values in metrics.items()
                        },
                        "config": config,
                    }
                    atomic_torch_save(best_state, best_model_path)'''
    assert old_best in train_text
    train_text = train_text.replace(old_best, new_best, 1)

    training_done_marker = "    # Training done, evaluate on evaluation set\n"
    assert training_done_marker in train_text
    save_epoch = '''        val_scores = {
            name: float(values["score"])
            for name, values in metrics.items()
        }
        epoch_state = {
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "epoch": epoch,
            "global_step": epoch * len(train_loader),
            "loss_stats": loss_stats,
            "spiders": spiders,
            "val_scores": val_scores,
            "config": config,
            "python_rng_state": random.getstate(),
            "numpy_rng_state": np.random.get_state(),
            "torch_rng_state": torch.get_rng_state(),
            "cuda_rng_state": (
                torch.cuda.get_rng_state_all()
                if torch.cuda.is_available() else None
            ),
        }
        epoch_path = epoch_checkpoint_dir / f"epoch_{epoch:02d}.pt"
        atomic_torch_save(epoch_state, epoch_path)
        main_logger.info(f"Saved full checkpoint: {epoch_path}")

'''
    train_text = train_text.replace(
        training_done_marker,
        save_epoch + training_done_marker,
        1,
    )

    old_load = "    best_checkpoint = torch.load(str(model_output_dir) + '/best_model.pt')"
    new_load = "    best_checkpoint = load_checkpoint(best_model_path)"
    assert old_load in train_text
    train_text = train_text.replace(old_load, new_load, 1)

    train_file.write_text(train_text)
    print("Patched WavCaps train.py: full epoch checkpoints + auto resume")
else:
    print("WavCaps train.py is already patched")

legacy_folder_name = '''    folder_name = '{}_lr_{}_batch_{}_seed_{}'.format(exp_name,
                                                     config["optim_args"]["lr"],
                                                     config["data_args"]["batch_size"],
                                                     config["seed"])'''
compact_folder_name = '''    folder_name = f"{exp_name}_seed_{config['seed']}"'''
if legacy_folder_name in train_text:
    train_text = train_text.replace(legacy_folder_name, compact_folder_name, 1)
else:
    assert compact_folder_name in train_text

device_log = "    main_logger.info(f'Process on {device_name}')"
bf16_guard = '''    if config["training"].get("precision") != "bf16":
        raise ValueError('OnomaCap training requires training.precision: "bf16".')
    if device != "cuda" or not torch.cuda.is_bf16_supported():
        raise RuntimeError("OnomaCap BF16 training requires a BF16 CUDA GPU.")
    main_logger.info(f'Process on {device_name}')'''
if "OnomaCap BF16 training requires" not in train_text:
    assert device_log in train_text
    train_text = train_text.replace(device_log, bf16_guard, 1)
train_file.write_text(train_text)

Patched WavCaps train.py: full epoch checkpoints + auto resume


In [ ]:
from pathlib import Path

HTSAT_DIR = (
    Path(WAVCAPS_DIR)
    / "captioning"
    / "pretrained_models"
    / "audio_encoder"
)
HTSAT_DIR.mkdir(parents=True, exist_ok=True)

HTSAT_CKPT = Path(
    "/content/drive/MyDrive/OnomaCap/pretrained/HTSAT.ckpt"
)

shutil.copy(HTSAT_CKPT, HTSAT_DIR / "HTSAT.ckpt")

PosixPath('/content/WavCaps/captioning/pretrained_models/audio_encoder/HTSAT.ckpt')

In [ ]:
bart_file = Path(WAVCAPS_DIR) / "captioning/models/bart_captioning.py"

text = bart_file.read_text()
text = text.replace(
    "BartForConditionalGeneration.from_config(bart_config)",
    "BartForConditionalGeneration(bart_config)",
)
bart_file.write_text(text)

6894

In [ ]:
from pathlib import Path
import sys

# OnomaCap 논문 지표만 계산: BLEU-1~4, METEOR, ROUGE-L
# CIDEr/SPICE/SPIDEr와 Stanford CoreNLP는 사용하지 않음.
eval_file = Path("/content/WavCaps/captioning/eval_metrics.py")
eval_file.write_text(r'''from pathlib import Path
import csv

from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.rouge.rouge import Rouge


def _read_rows(value):
    if isinstance(value, list):
        return [dict(row) for row in value]
    path = Path(value)
    with path.open(newline="") as stream:
        return list(csv.DictReader(stream))


def _pack_metric(score, per_sample, file_names):
    return {
        "score": float(score),
        "scores": {
            file_name: float(per_sample[index])
            for index, file_name in enumerate(file_names)
        },
    }


def evaluate_metrics(prediction_file, reference_file, nb_reference_captions=5):
    predictions = _read_rows(prediction_file)
    references = _read_rows(reference_file)
    reference_by_name = {row["file_name"]: row for row in references}
    predictions.sort(key=lambda row: row["file_name"])
    file_names = [row["file_name"] for row in predictions]
    assert all(name in reference_by_name for name in file_names)

    gts = {
        index: [
            reference_by_name[row["file_name"]][f"caption_{caption_index}"]
            for caption_index in range(1, nb_reference_captions + 1)
        ]
        for index, row in enumerate(predictions)
    }
    res = {
        index: [row["caption_predicted"]]
        for index, row in enumerate(predictions)
    }

    bleu_scores, bleu_per_sample = Bleu(4).compute_score(gts, res)
    rouge_score, rouge_per_sample = Rouge().compute_score(gts, res)
    meteor_scorer = Meteor()
    try:
        meteor_score, meteor_per_sample = meteor_scorer.compute_score(gts, res)
    finally:
        if hasattr(meteor_scorer, "close"):
            meteor_scorer.close()
        elif hasattr(meteor_scorer, "meteor_p"):
            meteor_scorer.meteor_p.terminate()

    metrics = {
        f"bleu_{index + 1}": _pack_metric(
            bleu_scores[index], bleu_per_sample[index], file_names
        )
        for index in range(4)
    }
    metrics["meteor"] = _pack_metric(
        meteor_score, meteor_per_sample, file_names
    )
    metrics["rouge_l"] = _pack_metric(
        rouge_score, rouge_per_sample, file_names
    )
    return metrics
''')

# WavCaps validate()가 SPIDEr/CIDEr를 요구하지 않도록 수정
pretrain_file = Path("/content/WavCaps/captioning/pretrain.py")
pretrain_text = pretrain_file.read_text()
old_metric_log = '''        spider = metrics['spider']['score']
        cider = metrics['cider']['score']

        eval_time = time.time() - start_time

        val_logger.info(f'Cider: {cider:7.4f}')
        val_logger.info(
            f'Spider score using beam search (beam size:{beam_size}): {spider:7.4f}, eval time: {eval_time:.1f}')

        if beam_size == 3 and (epoch % 5) == 0:
            for metric, values in metrics.items():
                val_logger.info(f'beam search (size 3): {metric:<7s}: {values["score"]:7.4f}')

        return metrics'''
new_metric_log = '''        eval_time = time.time() - start_time
        for metric, values in metrics.items():
            val_logger.info(
                f'beam search (size {beam_size}): {metric:<7s}: '
                f'{values["score"]:7.4f}'
            )
        val_logger.info(f'Evaluation time: {eval_time:.1f}')
        return metrics'''
if "metrics['spider']" in pretrain_text:
    assert old_metric_log in pretrain_text
    pretrain_text = pretrain_text.replace(old_metric_log, new_metric_log, 1)

old_train_forward = "        loss = model(audio, text)"
bf16_train_forward = '''        with torch.autocast(
            device_type="cuda", dtype=torch.bfloat16
        ):
            loss = model(audio, text)'''
if old_train_forward in pretrain_text:
    pretrain_text = pretrain_text.replace(
        old_train_forward, bf16_train_forward, 1
    )
else:
    assert bf16_train_forward in pretrain_text

old_generation = '''            output = model.generate(samples=audios,
                                    num_beams=beam_size)'''
bf16_generation = '''            with torch.autocast(
                device_type="cuda", dtype=torch.bfloat16
            ):
                output = model.generate(
                    samples=audios, num_beams=beam_size
                )'''
if old_generation in pretrain_text:
    pretrain_text = pretrain_text.replace(
        old_generation, bf16_generation, 1
    )
else:
    assert bf16_generation in pretrain_text
pretrain_file.write_text(pretrain_text)

# validation은 beam=3 한 번만 수행하고 BLEU-1로 best_model.pt 선정
# 모든 epoch checkpoint와 모든 논문 지표는 별도로 계속 저장됨.
train_file = Path("/content/WavCaps/captioning/train.py")
train_text = train_file.read_text()
validation_start = train_text.index("        # validation loop, validation after each epoch")
epoch_save_start = train_text.index("        val_scores = {", validation_start)
paper_validation = '''        # validation loop: OnomaCap paper metrics, beam=3
        main_logger.info("Validating OnomaCap metrics...")
        metrics = validate(
            val_loader,
            model,
            device=device,
            log_dir=log_output_dir,
            epoch=epoch,
            beam_size=3,
        )
        selection_score = metrics["bleu_1"]["score"]
        spiders.append(selection_score)
        wandb.log(
            {f"val/{name}": float(values["score"]) for name, values in metrics.items()}
            | {"epoch": epoch}
        )

        if selection_score >= max(spiders):
            best_state = {
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "beam_size": 3,
                "epoch": epoch,
                "selection_metric": "bleu_1",
                "selection_score": float(selection_score),
                "val_scores": {
                    name: float(values["score"])
                    for name, values in metrics.items()
                },
                "config": config,
            }
            atomic_torch_save(best_state, best_model_path)

'''
train_text = (
    train_text[:validation_start]
    + paper_validation
    + train_text[epoch_save_start:]
)
train_text = train_text.replace(
    'spiders = checkpoint.get("spiders", [])',
    'spiders = checkpoint.get("selection_scores", checkpoint.get("spiders", []))',
)
train_text = train_text.replace(
    '"spiders": spiders,',
    '"selection_scores": spiders,',
)

test_start = train_text.index("    # Training done, evaluate on evaluation set")
test_end = train_text.index("    main_logger.info('Evaluation done.')", test_start)
paper_test = '''    # Training done, evaluate best BLEU-1 checkpoint with paper metrics
    main_logger.info('Training done. Start evaluating OnomaCap metrics.')
    best_checkpoint = load_checkpoint(best_model_path)
    model.load_state_dict(best_checkpoint['model'])
    best_epoch = best_checkpoint['epoch']
    main_logger.info(f'Best BLEU-1 checkpoint occurred at epoch {best_epoch}.')
    test_metrics = validate(
        test_loader,
        model,
        device=device,
        log_dir=log_output_dir,
        epoch=0,
        beam_size=3,
    )
    wandb.log(
        {f"test/{name}": float(values["score"]) for name, values in test_metrics.items()}
    )
'''
train_text = train_text[:test_start] + paper_test + train_text[test_end:]
train_file.write_text(train_text)

os.environ["PYTHONPATH"] = "/content/WavCaps/captioning"
print(os.environ["PYTHONPATH"])

# 실제 OnomaCap validation 샘플로 논문 지표 smoke test
captioning_dir = Path("/content/WavCaps/captioning")
sys.path.insert(0, str(captioning_dir))
sys.modules.pop("eval_metrics", None)
sys.modules.pop("pretrain", None)

from eval_metrics import evaluate_metrics

smoke_records = json.loads(
    (captioning_dir / "data/OnomaCap/json_files/val.json").read_text()
)["data"][:4]
smoke_predictions = [
    {
        "file_name": Path(record["audio"]).name,
        "caption_predicted": record["caption_1"],
    }
    for record in smoke_records
]
smoke_references = []
for record in smoke_records:
    row = {"file_name": Path(record["audio"]).name}
    row.update({f"caption_{i}": record[f"caption_{i}"] for i in range(1, 6)})
    smoke_references.append(row)

smoke_scores = evaluate_metrics(smoke_predictions, smoke_references)
required_metric_names = {
    "bleu_1", "bleu_2", "bleu_3", "bleu_4",
    "meteor", "rouge_l",
}
missing_metrics = required_metric_names - set(smoke_scores)
assert not missing_metrics, f"Missing metrics: {sorted(missing_metrics)}"
print("OnomaCap metric smoke test passed:", sorted(smoke_scores))

/content/WavCaps/captioning
{'testlen': 24, 'reflen': 24, 'guess': [24, 20, 16, 12], 'correct': [24, 20, 16, 12]}
ratio: 0.9999999999583333
OnomaCap metric smoke test passed: ['bleu_1', 'bleu_2', 'bleu_3', 'bleu_4', 'meteor', 'rouge_l']


In [ ]:
# 본 학습 전 end-to-end smoke test: 실제 데이터 1배치만 사용
import gc
import ruamel.yaml as yaml

SMOKE_MARKER = Path("/content/onomacap_full_smoke_passed")
SMOKE_MARKER.unlink(missing_ok=True)
smoke_previous_cwd = Path.cwd()

try:
    os.chdir("/content/WavCaps/captioning")

    from data_handling.datamodule import AudioCaptionDataModule
    from models.bart_captioning import BartCaptionModel
    from pretrain import validate
    from tools.optim_utils import get_optimizer
    from tools.utils import setup_seed

    with open("settings/onomacap.yaml", "r") as stream:
        smoke_config = yaml.safe_load(stream)

    smoke_config["data_args"] = dict(smoke_config["data_args"])
    smoke_config["data_args"]["batch_size"] = 2
    smoke_config["data_args"]["num_workers"] = 0
    setup_seed(smoke_config["seed"])

    assert torch.cuda.is_available(), "CUDA GPU가 필요합니다."
    assert torch.cuda.is_bf16_supported(), "BF16 지원 CUDA GPU가 필요합니다."
    smoke_device = "cuda"

    smoke_data = AudioCaptionDataModule(smoke_config, "OnomaCap")
    smoke_train_loader = smoke_data.train_dataloader(is_distributed=False)
    smoke_val_loader = smoke_data.val_dataloader()

    smoke_model = BartCaptionModel(smoke_config).to(smoke_device)
    smoke_optimizer = get_optimizer(
        smoke_model.parameters(),
        lr=smoke_config["optim_args"]["lr"],
        betas=smoke_config["optim_args"]["betas"],
        eps=smoke_config["optim_args"]["eps"],
        momentum=smoke_config["optim_args"]["momentum"],
        weight_decay=smoke_config["optim_args"]["weight_decay"],
        optimizer_name=smoke_config["optim_args"]["optimizer_name"],
    )

    # 실제 forward/backward/optimizer step
    smoke_audio, smoke_text, _, _ = next(iter(smoke_train_loader))
    smoke_model.train()
    smoke_optimizer.zero_grad(set_to_none=True)
    with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        smoke_loss = smoke_model(smoke_audio.to(smoke_device), smoke_text)
    assert torch.isfinite(smoke_loss), f"Non-finite smoke loss: {smoke_loss}"
    smoke_loss.backward()
    torch.nn.utils.clip_grad_norm_(
        smoke_model.parameters(),
        smoke_config["training"]["clip_grad"],
    )
    smoke_optimizer.step()

    # 실제 beam generation + decode + OnomaCap 논문 metric 계산
    smoke_log_dir = Path("/content/onomacap_smoke_logs")
    smoke_log_dir.mkdir(parents=True, exist_ok=True)
    smoke_val_batch = next(iter(smoke_val_loader))
    smoke_metrics = validate(
        [smoke_val_batch],
        smoke_model,
        device=smoke_device,
        log_dir=smoke_log_dir,
        epoch=0,
        beam_size=3,
    )
    assert set(smoke_metrics) == {
        "bleu_1", "bleu_2", "bleu_3", "bleu_4", "meteor", "rouge_l"
    }

    # Drive checkpoint 경로 쓰기/읽기 점검
    smoke_checkpoint = Path(
        "/content/drive/MyDrive/OnomaCap/checkpoints/.smoke_test.pt"
    )
    smoke_checkpoint.parent.mkdir(parents=True, exist_ok=True)
    try:
        torch.save(
            {"epoch": 0, "probe": torch.tensor([2024])},
            smoke_checkpoint,
        )
        try:
            smoke_loaded = torch.load(
                smoke_checkpoint,
                map_location="cpu",
                weights_only=False,
            )
        except TypeError:
            smoke_loaded = torch.load(smoke_checkpoint, map_location="cpu")
        assert smoke_loaded["probe"].item() == 2024
    finally:
        smoke_checkpoint.unlink(missing_ok=True)

    SMOKE_MARKER.write_text("passed")
    print(
        "FULL SMOKE TEST PASSED |",
        f"loss={smoke_loss.item():.4f} |",
        f"BLEU-1={smoke_metrics['bleu_1']['score']:.4f} |",
        f"METEOR={smoke_metrics['meteor']['score']:.4f} |",
        f"ROUGE-L={smoke_metrics['rouge_l']['score']:.4f}",
    )
finally:
    os.chdir(smoke_previous_cwd)
    for smoke_name in [
        "smoke_model", "smoke_optimizer", "smoke_data",
        "smoke_train_loader", "smoke_val_loader",
        "smoke_audio", "smoke_loss", "smoke_val_batch",
    ]:
        if smoke_name in globals():
            del globals()[smoke_name]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
%cd /content/WavCaps/captioning
!PYTHONPATH=/content/caption-evaluation-tools:/content/WavCaps/captioning python train.py \
    --exp_name {EXPERIMENT_NAME} \
    --config settings/onomacap.yaml \
    --lr 3e-5 \
    --seed {MODEL_SEED}

In [ ]:
from pathlib import Path
import torch

folder_name = f"{EXPERIMENT_NAME}_seed_{MODEL_SEED}"
best_path = (
    Path("/content/drive/MyDrive/OnomaCap/checkpoints")
    / folder_name
    / "best_model.pt"
)

best = torch.load(
    best_path,
    map_location="cpu",
    weights_only=False,
)

print("best epoch:", best["epoch"])
print("selection metric:", best["selection_metric"])
print("best val BLEU-1:", best["selection_score"])
print("all val scores:", best["val_scores"])

best epoch: 13
selection metric: bleu_1
best val BLEU-1: 0.6653062712231312
all val scores: {'bleu_1': 0.6653062712231312, 'bleu_2': 0.5557577696904216, 'bleu_3': 0.45783205098720564, 'bleu_4': 0.36673539172880915, 'meteor': 0.25738260321315604, 'rouge_l': 0.5574722180096177}


In [ ]:
# 원하는 epoch checkpoint를 test set에서 평가
TEST_EPOCH = 10  # 이 숫자만 바꾸면 됨
TEST_BEAM_SIZE = 3

import gc
import ruamel.yaml as yaml
from loguru import logger

test_previous_cwd = Path.cwd()
test_model = None
try:
    captioning_dir = Path("/content/WavCaps/captioning")
    os.chdir(captioning_dir)
    if str(captioning_dir) not in sys.path:
        sys.path.insert(0, str(captioning_dir))

    # 평가 설정 셀에서 수정한 evaluator를 확실히 다시 로드
    sys.modules.pop("eval_metrics", None)
    sys.modules.pop("pretrain", None)
    from data_handling.datamodule import AudioCaptionDataModule
    from models.bart_captioning import BartCaptionModel
    from pretrain import validate

    with open("settings/onomacap.yaml", "r") as stream:
        test_config = yaml.safe_load(stream)
    test_config["seed"] = MODEL_SEED

    folder_name = f"{EXPERIMENT_NAME}_seed_{MODEL_SEED}"
    checkpoint_path = (
        Path("/content/drive/MyDrive/OnomaCap/checkpoints")
        / folder_name
        / "epochs"
        / f"epoch_{TEST_EPOCH:02d}.pt"
    )
    assert checkpoint_path.is_file(), f"Checkpoint not found: {checkpoint_path}"

    device = "cuda" if torch.cuda.is_available() else "cpu"
    test_data = AudioCaptionDataModule(test_config, "OnomaCap")
    test_loader = test_data.test_dataloader()
    test_model = BartCaptionModel(test_config).to(device)

    try:
        checkpoint = torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=False,
        )
    except TypeError:
        checkpoint = torch.load(checkpoint_path, map_location="cpu")
    test_model.load_state_dict(checkpoint["model"])
    del checkpoint
    gc.collect()

    result_dir = (
        Path("/content/drive/MyDrive/OnomaCap/results")
        / folder_name
        / f"test_epoch_{TEST_EPOCH:02d}"
    )
    result_dir.mkdir(parents=True, exist_ok=True)

    # 샘플별 예측/정답은 파일에만 남기고 Colab 화면에는 출력하지 않는다.
    logger.remove()
    evaluation_log_path = result_dir / "evaluation.log"
    logger.add(
        str(evaluation_log_path),
        level="INFO",
        format="{time:YYYY-MM-DD HH:mm:ss} | {message}",
    )
    try:
        test_metrics = validate(
            test_loader,
            test_model,
            device=device,
            log_dir=result_dir,
            epoch=TEST_EPOCH,
            beam_size=TEST_BEAM_SIZE,
        )
    finally:
        logger.remove()
        logger.add(sys.stderr, level="WARNING")
    test_scores = {
        name: float(values["score"])
        for name, values in test_metrics.items()
    }
    scores_path = result_dir / "scores.json"
    scores_path.write_text(
        json.dumps(
            {
                "epoch": TEST_EPOCH,
                "beam_size": TEST_BEAM_SIZE,
                "checkpoint": str(checkpoint_path),
                "test_scores": test_scores,
            },
            indent=2,
        )
    )

    score_table = pd.DataFrame.from_dict(
        test_scores, orient="index", columns=[f"test_epoch_{TEST_EPOCH}"]
    )
    display(score_table)
    print("checkpoint:", checkpoint_path)
    print("saved:", scores_path)
finally:
    os.chdir(test_previous_cwd)
    if test_model is not None:
        del test_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
100%|██████████| 50/50 [00:39<00:00,  1.27it/s]
